In [4]:
import pandas as pd
import numpy as np
import os

# =========================
# 1. Check CatBoost
# =========================

try:
    import catboost
    print("CatBoost installed: Yes")
    print("CatBoost version:", catboost.__version__)
except ImportError:
    print("CatBoost installed: No")
    print("Run: !pip install catboost")

# =========================
# 2. File paths
# =========================

train_path = "../MLOpsedian/data/processed/train_model_ready.csv"
test_path = "../MLOpsedian/data/processed/test_model_ready.csv"

print("\nFile check:")
print("Train file exists:", os.path.exists(train_path))
print("Test file exists:", os.path.exists(test_path))

# =========================
# 3. Load files
# =========================

train = pd.read_csv(train_path)
test = pd.read_csv(test_path)

print("\nDataset check:")
print("Train shape:", train.shape)
print("Test shape:", test.shape)

# =========================
# 4. Basic setup
# =========================

id_col = "record_id"
target = "flood_risk_score"

# =========================
# 5. Column alignment
# =========================

print("\nColumn alignment:")
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))

# =========================
# 6. Missing values
# =========================

print("\nMissing values:")
print("Train missing:", train.isnull().sum().sum())
print("Test missing:", test.isnull().sum().sum())

# =========================
# 7. Duplicate IDs
# =========================

print("\nDuplicate IDs:")
print("Train duplicate IDs:", train[id_col].duplicated().sum())
print("Test duplicate IDs:", test[id_col].duplicated().sum())

# =========================
# 8. Target check
# =========================

print("\nTarget check:")
print(train[target].describe())
print("Target valid between 0 and 1:", train[target].between(0, 1).all())

# =========================
# 9. Prepare features
# =========================

X = train.drop(columns=[id_col, target])
y = train[target]

X_test = test.drop(columns=[id_col])
test_ids = test[id_col]

print("\nFeature setup:")
print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)
print("Feature columns match:", list(X.columns) == list(X_test.columns))

# =========================
# 10. Categorical features
# =========================

cat_features = X.select_dtypes(include=["object", "str"]).columns.tolist()

print("\nCategorical features:")
print("Number of categorical features:", len(cat_features))
print(cat_features)

# =========================
# 11. Unseen categories check
# =========================

print("\nUnseen categories in test:")
for col in cat_features:
    train_values = set(X[col].astype(str).unique())
    test_values = set(X_test[col].astype(str).unique())
    
    unseen_in_test = test_values - train_values
    
    print(col, ":", len(unseen_in_test))

# =========================
# 12. Constant columns
# =========================

constant_train_cols = []
constant_test_cols = []

for col in X.columns:
    if X[col].nunique(dropna=False) <= 1:
        constant_train_cols.append(col)

for col in X_test.columns:
    if X_test[col].nunique(dropna=False) <= 1:
        constant_test_cols.append(col)

print("\nConstant columns:")
print("Constant columns in train:", constant_train_cols)
print("Constant columns in test:", constant_test_cols)

# =========================
# 13. Final summary
# =========================

print("\n===== CATBOOST PRECHECK SUMMARY =====")

print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Missing train:", train.isnull().sum().sum())
print("Missing test:", test.isnull().sum().sum())
print("Duplicate train IDs:", train[id_col].duplicated().sum())
print("Duplicate test IDs:", test[id_col].duplicated().sum())
print("Train-only columns:", set(train.columns) - set(test.columns))
print("Test-only columns:", set(test.columns) - set(train.columns))
print("X shape:", X.shape)
print("X_test shape:", X_test.shape)
print("Feature columns match:", list(X.columns) == list(X_test.columns))
print("Number of categorical features:", len(cat_features))
print("Target valid:", train[target].between(0, 1).all())

CatBoost installed: Yes
CatBoost version: 1.2.10

File check:
Train file exists: True
Test file exists: True

Dataset check:
Train shape: (19700, 69)
Test shape: (5300, 68)

Column alignment:
Train-only columns: {'flood_risk_score'}
Test-only columns: set()

Missing values:
Train missing: 0
Test missing: 0

Duplicate IDs:
Train duplicate IDs: 0
Test duplicate IDs: 0

Target check:
count    19700.000000
mean         0.478517
std          0.235776
min          0.000000
25%          0.341675
50%          0.474300
75%          0.613625
max          1.000000
Name: flood_risk_score, dtype: float64
Target valid between 0 and 1: True

Feature setup:
X shape: (19700, 67)
y shape: (19700,)
X_test shape: (5300, 67)
Feature columns match: True

Categorical features:
Number of categorical features: 14
['district', 'place_name', 'landcover', 'soil_type', 'water_supply', 'electricity', 'road_quality', 'urban_rural', 'water_presence_flag', 'flood_occurrence_current_event', 'is_good_to_live', 'reason_n

In [5]:
import pandas as pd
import numpy as np
import os

from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

from catboost import CatBoostRegressor

In [6]:
train = pd.read_csv("../MLOpsedian/data/processed/train_model_ready.csv")
test = pd.read_csv("../MLOpsedian/data/processed/test_model_ready.csv")

print("Train shape:", train.shape)
print("Test shape:", test.shape)

Train shape: (19700, 69)
Test shape: (5300, 68)


In [7]:
id_col = "record_id"
target = "flood_risk_score"

drop_cols = [
    "generation_date_missing"
]

train_cb = train.drop(columns=drop_cols, errors="ignore").copy()
test_cb = test.drop(columns=drop_cols, errors="ignore").copy()

X = train_cb.drop(columns=[id_col, target])
y = train_cb[target]

X_test = test_cb.drop(columns=[id_col])
test_ids = test_cb[id_col]

print("X shape:", X.shape)
print("y shape:", y.shape)
print("X_test shape:", X_test.shape)

print("Columns match:", list(X.columns) == list(X_test.columns))

X shape: (19700, 66)
y shape: (19700,)
X_test shape: (5300, 66)
Columns match: True


In [8]:
cat_features = [
    col for col in X.columns
    if X[col].dtype == "object" or str(X[col].dtype) == "str"
]

print("Number of categorical features:", len(cat_features))
print(cat_features)

Number of categorical features: 14
['district', 'place_name', 'landcover', 'soil_type', 'water_supply', 'electricity', 'road_quality', 'urban_rural', 'water_presence_flag', 'flood_occurrence_current_event', 'is_good_to_live', 'reason_not_good_to_live', 'is_synthetic', 'generation_date']


In [9]:
def validate_submission(submission):
    print("Submission shape:", submission.shape)
    print(submission.head())
    
    print("\nMissing values:")
    print(submission.isnull().sum())
    
    print("\nPrediction range:")
    print("Min:", submission["flood_risk_score"].min())
    print("Max:", submission["flood_risk_score"].max())
    
    assert submission.shape[0] == len(test_ids)
    assert list(submission.columns) == ["record_id", "flood_risk_score"]
    assert submission["flood_risk_score"].isnull().sum() == 0
    assert submission["flood_risk_score"].between(0, 1).all()
    
    print("\nSubmission is valid.")

In [10]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)

rmse_scores = []
mae_scores = []
r2_scores = []

test_preds = np.zeros(len(X_test))

for fold, (train_idx, valid_idx) in enumerate(kf.split(X), 1):
    print("\n" + "=" * 70)
    print(f"Fold {fold}")
    
    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]
    
    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.03,
        depth=6,
        loss_function="RMSE",
        eval_metric="RMSE",
        random_seed=42 + fold,
        verbose=200,
        early_stopping_rounds=150,
        allow_writing_files=False
    )
    
    model.fit(
        X_train,
        y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )
    
    valid_preds = model.predict(X_valid)
    valid_preds = np.clip(valid_preds, 0, 1)
    
    rmse = np.sqrt(mean_squared_error(y_valid, valid_preds))
    mae = mean_absolute_error(y_valid, valid_preds)
    r2 = r2_score(y_valid, valid_preds)
    
    rmse_scores.append(rmse)
    mae_scores.append(mae)
    r2_scores.append(r2)
    
    print("Fold RMSE:", rmse)
    print("Fold MAE :", mae)
    print("Fold R²  :", r2)
    
    fold_test_preds = model.predict(X_test)
    test_preds += fold_test_preds / kf.n_splits

print("\n" + "=" * 70)
print("CatBoost CV Results")
print("Average RMSE:", np.mean(rmse_scores))
print("Average MAE :", np.mean(mae_scores))
print("Average R²  :", np.mean(r2_scores))


Fold 1
0:	learn: 0.2348522	test: 0.2385527	best: 0.2385527 (0)	total: 88.4ms	remaining: 2m 12s
200:	learn: 0.2269318	test: 0.2347089	best: 0.2346924 (198)	total: 2.09s	remaining: 13.5s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.2346519192
bestIteration = 233

Shrink model to first 234 iterations.
Fold RMSE: 0.23465191929921855
Fold MAE : 0.17942606422168242
Fold R²  : 0.03354358745782904

Fold 2
0:	learn: 0.2362134	test: 0.2332313	best: 0.2332313 (0)	total: 16ms	remaining: 24s
200:	learn: 0.2283224	test: 0.2292169	best: 0.2292074 (194)	total: 1.88s	remaining: 12.2s
400:	learn: 0.2242945	test: 0.2291284	best: 0.2290819 (384)	total: 3.82s	remaining: 10.5s
Stopped by overfitting detector  (150 iterations wait)

bestTest = 0.2290818813
bestIteration = 384

Shrink model to first 385 iterations.
Fold RMSE: 0.22908188152088957
Fold MAE : 0.1744506972317693
Fold R²  : 0.03618823401311644

Fold 3
0:	learn: 0.2352486	test: 0.2369700	best: 0.2369700 (0)	total: 8.83ms	re

In [11]:
catboost_preds = np.clip(test_preds, 0, 1)

print("Prediction min:", catboost_preds.min())
print("Prediction max:", catboost_preds.max())
print("Prediction mean:", catboost_preds.mean())
print("Prediction std:", catboost_preds.std())

Prediction min: 0.36915170478834614
Prediction max: 0.6192474835142522
Prediction mean: 0.47800513902252784
Prediction std: 0.04119658968732579


In [12]:
sub_catboost = pd.DataFrame({
    "record_id": test_ids,
    "flood_risk_score": catboost_preds
})

validate_submission(sub_catboost)

sub_catboost.to_csv("../submissions/sub_v006_catboost_baseline.csv", index=False)

print("Saved: ../submissions/sub_v006_catboost_baseline.csv")

Submission shape: (5300, 2)
  record_id  flood_risk_score
0   F104559          0.463268
1   F100765          0.460346
2   F107573          0.491392
3   F110345          0.533002
4   F118850          0.497199

Missing values:
record_id           0
flood_risk_score    0
dtype: int64

Prediction range:
Min: 0.36915170478834614
Max: 0.6192474835142522

Submission is valid.
Saved: ../submissions/sub_v006_catboost_baseline.csv


In [13]:
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 19700 entries, 0 to 19699
Data columns (total 69 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   record_id                         19700 non-null  str    
 1   district                          19700 non-null  str    
 2   place_name                        19700 non-null  str    
 3   latitude                          19700 non-null  float64
 4   longitude                         19700 non-null  float64
 5   elevation_m                       19700 non-null  float64
 6   distance_to_river_m               19700 non-null  float64
 7   landcover                         19700 non-null  str    
 8   soil_type                         19700 non-null  str    
 9   water_supply                      19700 non-null  str    
 10  electricity                       19700 non-null  str    
 11  road_quality                      19700 non-null  str    
 12  population_dens

In [15]:
train.shape

(19700, 69)